
# Estimation of DWPA model parameters by grid search and gradient-based optimisation

The code below estimates parameters of the DWPA model using grid search and a
quasi-Newton optimisation method, LBFGS.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joacorapela/lds_itba_2026/blob/main/code/notebooks/exercises/plotMLlbfgs.ipynb)

## Import packages



In [1]:
import time
import math
import numpy as np
import torch
import subprocess
from packaging.version import Version
from tqdm import tqdm
import plotly.graph_objs as go

In [2]:
# Strip any local build suffixes (e.g., "+cu121") for clean comparison
current_version = torch.__version__.split("+")[0]

if Version(current_version) < Version("2.8.0"):
  print(f"Found PyTorch {torch.__version__}. Upgrading to the latest version...")
  subprocess.run(["pip", "install", "--upgrade", "torch"], check=True)
  print("Upgrade complete! Note: You may need to restart your runtime.")
else:
  print(f"PyTorch version {torch.__version__} is >= 2.8.0. No upgrade needed.")

PyTorch version 2.8.0+cu128 is >= 2.8.0. No upgrade needed.


## Function to compute the log-likelihood of LDS model parameters



In [3]:
def compute_log_like(y, A, Q, C, R, m0, V0):
    N = y.shape[1]
    P = C.shape[0]

    # Initial log-likelihood scalar/tensor
    log_like = torch.tensor(
        -0.5 * N * P * math.log(2 * math.pi),
        dtype=torch.double,
    )

    for n in range(N):
        if n == 0:
            # Out-of-place assignments for initial step
            xnn1 = A @ torch.unsqueeze(m0, -1)
            Pnn1 = A @ V0 @ A.T + Q
        else:
            xnn1 = A @ xnn
            Pnn1 = A @ Pnn @ A.T + Q

        Stmp = C @ Pnn1 @ C.T + R
        Sn = (Stmp + Stmp.T) / 2

        # Solve for inverse cleanly
        Sinv = torch.linalg.solve(
            Sn, torch.eye(P, dtype=torch.double),
        )

        K = Pnn1 @ C.T @ Sinv

        pred_obs = C @ xnn1

        # Handle observation slice dimensions safely
        yn = y[:, n:(n + 1)]  # keeps shape [P, 1]
        innov = yn - pred_obs

        xnn = xnn1 + K @ innov
        Pnn = Pnn1 - K @ C @ Pnn1

        # Out-of-place log-likelihood update
        log_like = log_like - 0.5 * (
            torch.logdet(Sn) + innov.T @ torch.linalg.solve(Sn, innov)
        )

    return log_like

## Set hyperparameters



In [4]:
pos_x0 = 0.0
vel_x0 = 0.0
ace_x0 = 0.0
dt = 1e-3
num_pos = 1000
sigma_a = 1.0
sigma_y = 1.0
sqrt_diag_V0_value = 1e-03

## Set LDS parameters



In [5]:
A = torch.tensor([[1, dt, dt**2/2],
                  [0, 1, dt],
                  [0, 0, 1]],
                 dtype=torch.double)
C = torch.tensor([[1, 0, 0]], dtype=torch.double)

Qe = torch.tensor([[dt**4/4, dt**3/2, dt**2/2],
                   [dt**3/2, dt**2,   dt],
                   [dt**2/2, dt,      1]],
                  dtype=torch.double)
Q = sigma_a**2 * Qe
R = torch.diag(torch.tensor([sigma_y**2]))
m0 = torch.tensor([pos_x0, vel_x0, ace_x0],
                 dtype=torch.double)
V0 = torch.diag(torch.ones_like(m0)*sqrt_diag_V0_value**2)

## Sample from the LDS



In [6]:
start_time = time.time()

# fix random seed
np.random.seed(seed=1234)

M = A.shape[0]
P = C.shape[0]

# state noise
w = np.random.multivariate_normal(np.zeros(M), Q.numpy(), num_pos).T
# measurement noise
v = np.random.multivariate_normal(np.zeros(P), R.numpy(), num_pos).T
# initial state noise
x0 = np.random.multivariate_normal(m0.numpy(), V0.numpy(), 1).flatten()
# sample
x = np.empty(shape=(M, num_pos))
y = np.empty(shape=(P, num_pos))
x[:, 0] = A.numpy() @ x0 + w[:, 0]
for n in range(1, num_pos):
    x[:, n] = A.numpy() @ x[:, n-1] + w[:, n]
y = C.numpy() @ x + v
t = np.arange(num_pos) * dt

elapsed_time = time.time() - start_time

y = torch.from_numpy(y)
t = torch.from_numpy(t)

## Run a quasi-Newton (LBFGS) maximisation



In [ ]:
# Define variables that control the optimisation

max_epochs = 50
max_iter_per_epoch = 10
lr = .05
tolerance_grad = 1e-7
tolerance_change = 1e-9
# line_search_fn = "strong_wolfe"
line_search_fn = None

# Set initial conditions
sigma_a0 = 39.0
sigma_y0 = 1.1

# perform optimisation
optim_params = {
    "lr": lr,
    "max_iter": max_iter_per_epoch,
    "tolerance_grad": tolerance_grad,
    "tolerance_change": tolerance_change,
    "line_search_fn": line_search_fn,
}
sigma_a_tensor = torch.tensor([sigma_a0], dtype=torch.double)
sigma_y_tensor = torch.tensor([sigma_y0], dtype=torch.double)
sigma_a_tensor.requires_grad = True
sigma_y_tensor.requires_grad = True
optimizer = torch.optim.LBFGS([sigma_a_tensor, sigma_y_tensor], **optim_params)


def closure():
    optimizer.zero_grad()

    R_lbfgs = (sigma_y_tensor**2).reshape(P, P)
    Q_lbfgs = sigma_a_tensor**2 * Qe
    log_like = compute_log_like(
        y=y, A=A, Q=Q_lbfgs, C=C, R=R_lbfgs, m0=m0, V0=V0
    )
    loss = -log_like
    loss.backward(retain_graph=True)
    return loss


# Run LBFGS optimisation
print(f"{'Iteration':<10} | {'Loss (Neg Log-Likelihood)':<25} | {'sigma_a':<12} | {'sigma_y':<12}")
print("-" * 65)

R_lbfgs = (sigma_y_tensor**2).reshape(P, P)
Q_lbfgs = sigma_a_tensor**2 * Qe
ll0 = compute_log_like(
    y=y, A=A, Q=Q_lbfgs, C=C, R=R_lbfgs, m0=m0, V0=V0
)

hist_ll = [ll0]
hist_sigma_a = [sigma_a0]
hist_sigma_y = [sigma_y0]
for epoch in range(max_epochs):
    # Perform optimization step
    optimizer.step(closure)

    # Save history of parameters and log-likelihoods
    R_lbfgs = (sigma_y_tensor**2).reshape(P, P)
    Q_lbfgs = sigma_a_tensor**2 * Qe
    lln = compute_log_like(
        y=y, A=A, Q=Q_lbfgs, C=C, R=R_lbfgs, m0=m0, V0=V0
    )

    hist_ll.append(lln)
    hist_sigma_a.append(sigma_a_tensor.item())
    hist_sigma_y.append(sigma_y_tensor.item())

    # Print progress metrics
    print(f"{epoch+1:<10} | {lln.item():<25.4f} |"
          f" {sigma_a_tensor.item():<12.4f} | {sigma_y_tensor.item():<12.4f}")

Iteration  | Loss (Neg Log-Likelihood) | sigma_a      | sigma_y     
-----------------------------------------------------------------


/home/rapela/anaconda3/envs/lds_itba_2026/lib/python3.9/site-packages/torch/optim/lbfgs.py:457: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  loss = float(closure())


1          | -1426.9432                | 38.9996      | 1.0201      
2          | -1425.8665                | 38.9989      | 1.0025      
3          | -1425.5109                | 38.9980      | 0.9928      
4          | -1425.3890                | 38.9970      | 0.9872      
5          | -1425.3463                | 38.9959      | 0.9839      
6          | -1425.3311                | 38.9948      | 0.9819      
7          | -1425.3255                | 38.9937      | 0.9808      
8          | -1425.3233                | 38.9925      | 0.9800      
9          | -1425.3223                | 38.9911      | 0.9795      
10         | -1425.3218                | 38.9894      | 0.9792      
11         | -1425.3212                | 38.9866      | 0.9788      
12         | -1425.3204                | 38.9809      | 0.9784      


## Build grid of log-likelihood values



In [ ]:
sigma_a_min_seg1 = 0.1
sigma_a_max_seg1 = 2.0
sigma_a_step_seg1 = 0.1
sigma_a_min_seg2 = 3.0
sigma_a_max_seg2 = 40.0
sigma_a_step_seg2 = 1.0
sigma_y_min = 0.9
sigma_y_max = 1.1
sigma_y_step = 0.01
sigma_a_values = torch.concatenate([torch.arange(sigma_a_min_seg1, sigma_a_max_seg1,
                                                 sigma_a_step_seg1),
                                    torch.arange(sigma_a_min_seg2, sigma_a_max_seg2,
                                                 sigma_a_step_seg2)])
sigma_y_values = torch.arange(sigma_y_min, sigma_y_max, sigma_y_step)
ll_gs = torch.empty(size=(len(sigma_a_values), len(sigma_y_values)),
                dtype=torch.double)
for i, sigma_a_value in enumerate(
    tqdm(sigma_a_values, desc="Computing LL grid")
):
    Q_gs = Qe * sigma_a_value**2
    for j, sigma_y_value in enumerate(sigma_y_values):
        R_gs = torch.diag(torch.tensor([sigma_y_value**2], dtype=torch.double))
        ll_gs[i, j] = compute_log_like(y=y, A=A, Q=Q_gs, C=C, R=R_gs, m0=m0,
                                       V0=V0)

## Find grid maximum values



In [ ]:
ll_gs_argmax_flat = torch.argmax(ll_gs)
ll_gs_argmax = torch.unravel_index(ll_gs_argmax_flat, ll_gs.shape)
ll_gs_max = ll_gs[ll_gs_argmax]
max_sigma_a_gs = sigma_a_values[ll_gs_argmax[0]]
max_sigma_y_gs = sigma_y_values[ll_gs_argmax[1]]

## Plot log-likelihoods



In [ ]:
fig = go.Figure()

# Contour plot of log-likelihoods
trace = go.Contour(
    x=sigma_y_values,
    y=sigma_a_values,
    z=ll_gs,
    colorbar=dict(title=dict(text="Log-Likelihood")),
)
fig.add_trace(trace)

# Grid search maximum cross
trace = go.Scatter(
    x=[max_sigma_y_gs],
    y=[max_sigma_a_gs],
    mode="markers",
    marker_color="red",
    marker_size=10,
    marker_symbol="cross",
    hovertemplate=(
        r"$\sigma_ys$: %{x:.4f}<br>"
        r"$\sigma_a$: %{y:.4f}<br>"
        r"ll: %{text}<extra></extra>"
    ),
    text=[f"{ll_gs_max:.4f}"],
    showlegend=False,
)
fig.add_trace(trace)

# LBFGS optimisation path
trace = go.Scatter(
    x=hist_sigma_y,
    y=hist_sigma_a,
    mode="lines+markers",
    marker_color="green",
    hovertemplate=(
        r"$\sigma_y$: %{x:.4f}<br>"
        r"$\sigma_a$: %{y:.4f}<br>"
        r"%{text}<extra></extra>"
    ),
    text=[f"ll: {ll_item.item():.4f}" for ll_item in hist_ll],
    showlegend=False,
)
fig.add_trace(trace)

fig.update_yaxes(title=r"$\sigma_a$", title_font_size=20)
fig.update_xaxes(title=r"$\sigma_y$", title_font_size=20)
fig